# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install 'crewai[tools]' -q

In [3]:
!pip install youtube-transcript-api -q

In [4]:
!pip install langchain_openai -q

In [5]:
# Retrieve the API keys securely from Google Colab's user data
from google.colab import userdata
openai_api_key = userdata.get('ai_agents_openai')
serper_api_key = userdata.get('serper_api')

In [6]:
import os
# Set the API keys as environment variables for accessibility
os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['SERPER_API_KEY'] = serper_api_key

In [7]:
from crewai import Agent, Crew, Process, Task
from langchain_openai import ChatOpenAI
from IPython.display import display, Markdown
from crewai_tools import ScrapeWebsiteTool, SerperDevTool, YoutubeChannelSearchTool, YoutubeVideoSearchTool
from pydantic import BaseModel
import nest_asyncio

In [8]:
# Define the LLM
llm = ChatOpenAI(model_name = "gpt-4o", temperature = 0.5)

# Data Inputs

In [9]:
brand_info = {
    "name": "Zero to Mastery",
    "description": "An online tech academy that offers comprehensive courses and a community for learners seeking in-demand software and tech skills.",
    "brand_values": "Quality education, community-driven growth, career-focused learning, and up-to-date tech stacks.",
    "ideal_partner": "A YouTube channel focused on programming, software engineering, or tech career advice with an audience of motivated learners."
}

In [10]:
name = input("Enter name of Youtuber:") # ex: Theo - t3․gg

Enter name of Youtuber:Theo - t3․gg


In [11]:
channel_url = input("Enter the url for the Youtube channel:") # ex: https://www.youtube.com/@t3dotgg

Enter the url for the Youtube channel:https://www.youtube.com/@t3dotgg


In [12]:
# Define a single YouTube lead
youtube_lead = {
    "name": name,
    "channel_url": channel_url
}

# Research

In [ ]:
# Build a Research Agent
research_agent = Agent(
    role = "Influencer researcher",
    goal = f"Research about {youtube_lead['name']}'s channel {youtube_lead['channel_url']} to assess their suitability for {brand_info['name']}",
    backstory=f"You are an expert in Influencer partnerships that align with {brand_info['name']} and its respective values: {brand_info['brand_values']}",
    tools=[SerperDevTool(), ScrapeWebsiteTool(), YoutubeChannelSearchTool(), YoutubeVideoSearchTool()],
    verbose=True
)

In [15]:
# Define the google research task
channel_research_task = Task(
    description = f"""
    Research the channel {youtube_lead['name']} at {youtube_lead['channel_url']}:
    1. Identify channel focus and content type.
    2. Determine approximate subscriber count and engagement.
    3. Check for audience alignment with {brand_info['name']}.""",
    expected_output = "A research report on the channel's key attributes and audience",
    agent = research_agent
)

In [16]:
# Define the research videos task
videos_research_task = Task(
    description = f"""
    Research the most popular videos from channel {youtube_lead['name']} at {youtube_lead['channel_url']}.
    Summarize their topics and analyze the audiences.""",
    expected_output="A research report on the most popular videos and their topics",
    agent = research_agent)

In [17]:
# Define a sentiment on the channel via Google searches
sentiment_task = Task(
    description = f"""
    Conduct a sentiment analysis via Google Search on the channel {youtube_lead['name']} at {youtube_lead['channel_url']}.
    Look for user feedback, reviews, and discussions that indicate general sentiment.
    Search, for instance, on reddit.""",
    expected_output= "A sentiment analysis report on the channel",
    agent = research_agent)

# AI Scoring Agent

In [18]:
# Developping the Scoring Agent
scoring_agent =  Agent(
    role = "Partnership Scoring Agent",
    goal = f"""Score the partnership between {brand_info['name']} and {youtube_lead['name']}.
    Assess based on the {brand_info['brand_values']}""",
    backstory = f"You are an expert in Influencer Marketing working for {brand_info['name']}",
)

In [19]:
# Create a Base Model Class to provide a score
class Score(BaseModel):
    score: int
    justification: str

In [20]:
# Define the Scoring Task
scoring_task = Task(
    description = f"""
    Score the partnership between {brand_info['name']} and {youtube_lead['name']}""",
    expected_output = "A score between 1 and 10 and justification for the scoring. An explanation of the score indicating what fits the brand and what does not",
    agent = scoring_agent,
    context = [channel_research_task, videos_research_task, sentiment_task],
    output_pydantic= Score
)

# Build the AI Crew

In [22]:
nest_asyncio.apply()

# Assemble the AI crew
crew = Crew(
    agents = [research_agent, scoring_agent],
    tasks = [channel_research_task, videos_research_task, sentiment_task, scoring_task],
    process = Process.sequential,
    llm = llm,
    verbose = True,
    memory = True
)

result = await crew.kickoff_async(inputs=youtube_lead)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: fb07ff46-a439-4f98-bdd7-b352ed83cc4b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Research the channel Theo - t3․gg at https://www.youtube.com/@t3dotgg:                                     │
│      1. Identify channel focus and content type.                                                                │
│      2. Determine approximate subscriber count and engagement.                                                  │
│      3. Check for audience alignment with Zero to Mastery.                                                      │
│  ID: f5a315c5-0cfc-4aea-ab79-0beb114173b1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Influencer researcher                                                                                   │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Research the channel Theo - t3․gg at https://www.youtube.com/@t3dotgg:                                     │
│      1. Identify channel focus and content type.                                                                │
│      2. Determine approximate subscriber count and engagement.                                                  │
│      3. Check for audience alignment with Zero to Mastery.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'Theo t3.gg YouTube channel subscribers views 2025 @t3dotgg'}                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_website_content                                                                                     │
│  Args: {'website_url': 'https://www.youtube.com/@t3dotgg'}                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['Zero to Mastery influencer partnership audience goals values criteria', 'Theo t3.gg        │
│  previous influencer research']}                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (111 outputs left out)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Partnership Scoring Agent                                                                               │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Score the partnership between Zero to Mastery and Theo - t3․gg                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Partnership Scoring Agent                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"score":8,"justification":"Theo is a strong fit for Zero to Mastery’s developer-focused audience. His         │
│  practical, long-form tutorials and project builds align with quality education and career-focused learning,    │
│  while his regular coverage of React, Next.js, TypeScript, and AI-assisted development is especially well       │
│  matched to ZTM’s emphasis on up-to-date tech stacks. His established developer audience also offers good       │
│  potential for community-driven growth. The fit is not perfect: his channel includes opinionated commentary as  │
│  well as instruction, and public discussions show polarized reactions, including concerns about presentation,   │
│  AI hype, and perceived bias. Public reach figures also do not establish learner intent or conversion           │
│  potential. A focused, technically grounded partnership tied to a clear learning outcome is recommended, with   │
│  recent performance and audience data validated before committing."}                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Score the partnership between Zero to Mastery and Theo - t3․gg                                             │
│  Agent: Partnership Scoring Agent                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: fb07ff46-a439-4f98-bdd7-b352ed83cc4b                                                                       │
│  Final Output: {"score":8,"justification":"Theo is a strong fit for Zero to Mastery’s developer-focused         │
│  audience. His practical, long-form tutorials and project builds align with quality education and               │
│  career-focused learning, while his regular coverage of React, Next.js, TypeScript, and AI-assisted             │
│  development is especially well matched to ZTM’s emphasis on up-to-date tech stacks. His established developer  │
│  audience also offers good potential for community-driven growth. The fit is not perfect: his channel includes  │
│  opinionated commentary as well as instruction, and public discussions show polarized reactions, including      │
│  concerns about presentation, AI hype, and perceived bias. Public reach figures also do not establish learner   │
│  intent or conversion potential. A focused, technically grounded partnership tied to a clear learning outcome   │
│  is recommended, with recent performance and audience data validated before committing."}                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [23]:
import json

# Accessing the final outputs
channel_research_output = result.tasks_output[0].raw
videos_research_output = result.tasks_output[1].raw
sentiment_output = result.tasks_output[2].raw
scoring_output = result.tasks_output[3].raw

print("Channel Research Output:")
display(Markdown(channel_research_output))

print("\nVideos Research Output:")
display(Markdown(videos_research_output))

print("\nSentiment Analysis Output:")
display(Markdown(sentiment_output))

print("\nScoring Output:")
display(Markdown(scoring_output))

Channel Research Output:


# Influencer Research Report: Theo — t3․gg

**Channel:** [Theo - t3․gg on YouTube](https://www.youtube.com/@t3dotgg)  
**Creator:** Theo Browne, known online as Theo or t3  
**Assessment:** **Strong audience and topic fit for Zero to Mastery**, especially for developer-focused learning and current AI/software tooling.

## 1. Channel focus and content type

Theo describes his channel as focused on **software development and AI**. His content is aimed primarily at developers and people interested in building software, with recurring coverage of:

- **Web development and modern JavaScript/TypeScript tools**, including React and the T3 Stack
- **AI-assisted software development** and changes to developer workflows
- **Developer tools and product-building**, informed by his own work as a developer and founder
- **Commentary on the software industry and technology trends**

The channel combines practical coding and longer-form tutorials with opinions, analysis, and discussion of new tools. The public channel listing also includes long tutorial-style videos, such as *From 0 to Production – The Modern React Tutorial*. The channel is therefore not purely a structured course channel: its style is a creator-led mix of technical education, project-building, and tech commentary.

## 2. Approximate subscribers and engagement

- **Subscribers:** Approximately **578,000** according to the YouTube search listing available during research. Third-party listings show nearby counts in the mid-to-high 500,000s; counts fluctuate, so **about 575,000–580,000** is a reasonable estimate.
- **Channel library:** Approximately **1,100+ videos**.
- **Views and engagement:** Public search results show substantial reach, including recent videos with view counts in the **hundreds of thousands** and some listed examples exceeding **one million**. A third-party channel-statistics result reports roughly **116 million total views** (another listing gives a higher figure, likely reflecting different update times or counting methods).

A precise engagement rate cannot be established from the available public snippets alone. YouTube does not provide the channel’s full likes, comments, and audience-retention analytics publicly, and views vary considerably by video. As a practical partnership signal, the channel has a large developer-specific audience and evidence that individual uploads can reach well beyond its subscriber count.

## 3. Audience alignment with Zero to Mastery

**Alignment: High.**

The channel’s likely core audience—software developers, aspiring developers, and tech-oriented builders—overlaps closely with Zero to Mastery’s career-focused learners. Its coverage of web development, TypeScript, React, AI-assisted coding, and evolving developer tools also supports ZTM’s emphasis on current, practical technology stacks.

| ZTM value | Fit | Rationale |
|---|---|---|
| **Quality education** | **Good** | Theo publishes technical explainers and tutorial-length content, though the channel also includes commentary rather than following a consistently structured curriculum. |
| **Community-driven growth** | **Good** | The creator has an established developer presence and community ecosystem across social platforms and products; a partnership could connect ZTM with an already engaged software audience. |
| **Career-focused learning** | **Strong** | The subject matter is directly relevant to people building software skills and keeping up with practical development workflows. |
| **Up-to-date tech stacks** | **Very strong** | Frequent attention to current web-development tools and AI’s impact on coding closely matches ZTM’s focus on modern, relevant skills. |

## Partnership recommendation

Theo is a **strong potential partner** for campaigns aimed at aspiring or working developers, particularly those centered on modern web development, TypeScript/React, AI-enabled coding workflows, or building career-relevant software skills. The clearest fit would be a practical integration tied to a concrete learning outcome—such as helping viewers build foundational skills or progress from experimenting with tools to developing production-ready applications.

For a paid placement, validate current average views and engagement using recent long-form uploads and request the creator’s media kit or audience demographics. Public subscriber and view counts indicate scale, but do not establish the audience’s geographic mix, learner intent, or conversion performance.


Videos Research Output:


# Influencer Research Report: Theo — t3․gg

**Channel:** [Theo - t3․gg on YouTube](https://www.youtube.com/@t3dotgg)  
**Creator:** Theo Browne, known as Theo or t3  
**Focus:** Software development, web technologies, AI, and developer culture

## Research scope and method

This report reviews prominent high-view videos identified in public YouTube listings and search results. View counts are approximate snapshots and change over time. The list is a selection of highly viewed examples—not a definitive, continuously updated ranking of every upload.

## Most popular videos and what they cover

| Video | Approx. views found | Topic and audience appeal |
|---|---:|---|
| [PirateSoftware is right, this needs to stop](https://www.youtube.com/watch?v=A-IJ5QmeXpk) | **About 1 million** | A commentary video responding to a discussion associated with PirateSoftware. Its broad framing brings in developers and game-industry followers interested in online debate, work and career claims, and creator commentary—not only viewers seeking coding instruction. |
| [My Biggest Tutorial Ever (Build A FULL Google Drive Clone with React, Next, TypeScript and more)](https://www.youtube.com/watch?v=c-hKSbzooAg) | **About 774,000** | A long, hands-on project tutorial building a substantial application. It is aimed at learners who want to see how React, Next.js, and TypeScript fit together in a practical product build, as well as developers looking for implementation ideas. |
| [From 0 to Production – The Modern React Tutorial (RSCs, Next.js, Shadui, Drizzle, TS and more)](https://www.youtube.com/watch?v=d5x0JCZbAJs) | **About 583,000** | A roughly three-hour tutorial covering a modern full-stack React workflow, including React Server Components, Next.js, UI tooling, Drizzle, and TypeScript. This is the clearest match for viewers seeking structured, career-relevant technical learning. |
| [Agents Sucked At UI (Until Today)](https://www.youtube.com/watch?v=xq0eB8vxeqk) | **About 400,000+** in indexed results | Discusses AI agents’ ability to produce user interfaces and how newer approaches or tools may improve their output. It attracts developers experimenting with AI-assisted coding, product builders, and tech followers tracking agent capabilities. Counts vary across search snapshots. |
| [Why is every React site so slow?](https://www.youtube.com/watch?v=INLq9RPAYUw) | **About 301,000** | Examines performance problems in React applications. The audience is likely web developers evaluating framework choices, diagnosing front-end performance, or trying to make better engineering decisions. |
| [Did AI Kill React Native?](https://www.youtube.com/watch?v=GoDXfzpYGTc) | **About 252,000** in indexed recommendations | Considers how AI changes the case for React Native and mobile app development. It appeals to mobile and web developers, plus people assessing whether AI changes the value of particular skills or stacks. |

*Counts are approximate and reflect publicly indexed results; they may not match the current YouTube count.*

## What the popular videos suggest about the audience

### 1. A core audience of working and aspiring developers

The tutorial videos show strong demand for practical, project-oriented learning. Their subject matter—React, Next.js, TypeScript, and building complete applications—suggests viewers who want to learn by making software, rather than only watching general technology news.

### 2. Strong interest in current tools and stack decisions

Videos about React performance, React Native, and modern full-stack workflows indicate an audience attentive to current web-development practices and technical tradeoffs. This audience likely values timely explanations of what tools do, when to use them, and how they affect real projects.

### 3. AI-minded builders alongside traditional software learners

AI agents and AI’s effect on development are prominent topics. Viewers likely include developers integrating AI into existing workflows, builders prototyping products, and people trying to understand how AI may alter the skills expected in software careers.

### 4. Commentary expands reach beyond tutorial seekers

The high-view PirateSoftware-related video illustrates that Theo’s audience is not exclusively seeking lessons. Commentary and debate can reach a broader technology audience interested in online software culture, industry claims, and career conversations. This wider reach can be useful for awareness, but it does not by itself establish that all viewers are actively looking for structured education.

## Fit with Zero to Mastery

**Overall fit: Strong**, especially for campaigns aimed at developers and aspiring software engineers.

- **Quality education — Good fit:** The long-form tutorials demonstrate substantial technical teaching and practical project work. The broader channel also includes commentary, so not every popular upload is instructional.
- **Community-driven growth — Good potential:** A creator-led developer channel can introduce learning opportunities to an established technical audience. Audience community activity and learner intent should still be validated during partnership discussions.
- **Career-focused learning — Strong fit:** Full application builds and explanations of tools used in software development are relevant to learners building job-ready skills.
- **Up-to-date tech stacks — Very strong fit:** React, Next.js, TypeScript, and AI-assisted development align closely with a focus on contemporary tools and workflows.

## Partnership implications

The best alignment is likely a concrete learning-focused integration: for example, connecting a tutorial or product-building discussion to a clear progression from learning modern web fundamentals to building a production-ready application. AI-related placements could also fit well if framed around sound engineering practice and durable skills, rather than suggesting that tools alone replace learning.

The channel’s mix of instruction and commentary matters when selecting a placement. Tutorials are the clearest environment for an education-focused message; commentary videos may offer broader awareness but potentially weaker learner intent.

Before committing to a paid partnership, request current long-form average views, audience demographics, and any available campaign conversion results. Public view counts show reach, but do not establish audience geography, career stage, or willingness to enroll in a course.


Sentiment Analysis Output:


# Sentiment Analysis Report: Theo — t3․gg

**Channel:** [Theo - t3․gg on YouTube](https://www.youtube.com/@t3dotgg)  
**Creator:** Theo Browne  
**Research method:** Google Search focused on Reddit and developer-community discussions  
**Overall sentiment:** **Mixed, with notably polarized reactions**

## Overall read

Search results show both an audience that finds Theo’s developer content useful and a vocal group that distrusts or dislikes his presentation and opinions. Positive reactions tend to focus on practical web-development tutorials, project builds, and timely coverage of developer tools and AI. Negative reactions more often target his commentary style, perceived promotion or bias, and concerns about the accuracy or depth of some claims.

This is a qualitative read of publicly indexed discussions—not a representative survey of his viewers. Reddit threads can overrepresent strong opinions, and some results are from communities already discussing creator controversies.

## Positive feedback

- **Useful, current technical content:** Some commenters say they value Theo’s React, Next.js, TypeScript, and T3-related tutorials. Search results also surfaced recommendations of his channel and tutorial content in Next.js learning discussions.
- **Practical project-based learning:** Viewers looking for app-building videos mention Theo alongside other creators who make hands-on development content. One user discussing a first web project said Theo’s videos helped them learn concepts such as Next.js, single-page applications, and static-site generation.
- **Timely tool and AI coverage:** Some viewers appreciate his updates on AI tools, comparisons, and in-depth discussion of new features and developer workflows.

## Negative feedback

- **Presentation and personality:** A recurring criticism describes his delivery as smug, arrogant, or overly combative. Some comments object to the confidence and tone of his opinions, rather than to a specific tutorial.
- **Clickbait and emphatic commentary:** Search results include complaints about exaggerated titles and videos that viewers feel spend too long on commentary without delivering enough substance.
- **Perceived AI hype or promotion:** Some commenters characterize his AI coverage as “shilling” or believe he promotes tools and trends too enthusiastically. Others raise concerns that his business interests or investments could affect how he discusses products. These are audience perceptions, not independently established findings.
- **Questions about technical accuracy and tutorial reliability:** A subset of discussions questions the depth or accuracy of particular claims and points to issues encountered while following tutorials. These criticisms are specific to individual topics or videos and should not be treated as evidence that the channel’s overall instruction is unreliable.

## Sentiment by theme

| Theme | Audience reaction seen in search results |
|---|---|
| Modern web-development tutorials | **Positive, with some reservations** |
| AI and new-tool coverage | **Mixed:** valued by some, viewed as hype by others |
| Commentary and opinions | **Polarized; criticism is prominent in critical threads** |
| Trust and perceived bias | **Concern raised by some commenters** |
| Learning value overall | **Positive for some practical use cases, but not universally endorsed** |

## Assessment for Zero to Mastery

Theo remains a **strong subject-matter fit** for ZTM’s developer audience: his content overlaps with modern web development, practical software building, and changing developer tools. The sentiment findings add an important qualification: his audience may respond well to a concrete, technically grounded learning offer, while his opinionated style and the concerns about promotion could make audience trust especially important.

**Partnership recommendation:** Consider him for a focused campaign tied to a clear learner outcome—such as building a production-ready web application or developing durable skills alongside AI tools. Prioritize a tutorial or practical project context over a broad promotional message. Before committing, review recent video comments and performance, and discuss how the placement will be presented so the educational value is clear and the message feels credible.

**Confidence:** Moderate for identifying recurring praise and criticism in search-indexed discussions; low for estimating how common each view is across Theo’s full audience.


Scoring Output:


{"score":8,"justification":"Theo is a strong fit for Zero to Mastery’s developer-focused audience. His practical, long-form tutorials and project builds align with quality education and career-focused learning, while his regular coverage of React, Next.js, TypeScript, and AI-assisted development is especially well matched to ZTM’s emphasis on up-to-date tech stacks. His established developer audience also offers good potential for community-driven growth. The fit is not perfect: his channel includes opinionated commentary as well as instruction, and public discussions show polarized reactions, including concerns about presentation, AI hype, and perceived bias. Public reach figures also do not establish learner intent or conversion potential. A focused, technically grounded partnership tied to a clear learning outcome is recommended, with recent performance and audience data validated before committing."}